### Step 1: Setup Paths and Load Inputs
Loads the Step-2 wide-format claims, the medication claim log, the medication master (drug → classification), and the complications master (ICD-10 → systemic category). The two master files' headers are stripped of incidental whitespace so they can be merged safely downstream.

In [1]:
import pandas as pd
import numpy as np
import os

# 1. SETUP PATHS
current_dir = os.getcwd()
data_dir = os.path.join(current_dir, "..", "csv")

# Input Files
path_step2 = os.path.join(data_dir, "step2_claims_wide_format.xlsx")
path_meds = os.path.join(data_dir, "medication_data.xlsx")
path_med_master = os.path.join(data_dir, "Medication Master.xlsx")
path_comp_master = os.path.join(data_dir, "diabetes_list_of_complications.xlsx")

# Output File
output_file = os.path.join(data_dir, "step3_claims_meds_and_comps.xlsx")

print("--- STEP 3: FEATURE ENGINEERING ---")

# 2. LOAD DATA
if os.path.exists(path_step2):
    print("Loading datasets...")
    df_claims = pd.read_excel(path_step2)
    df_meds = pd.read_excel(path_meds)
    df_med_master = pd.read_excel(path_med_master, sheet_name="Sheet1")
    df_comp_master = pd.read_excel(path_comp_master, sheet_name="Sheet1")

    # Clean Headers (Remove extra spaces like ' drug_classification ')
    df_med_master.columns = [str(c).strip() for c in df_med_master.columns]
    df_comp_master.columns = [str(c).strip() for c in df_comp_master.columns]

    print(f" Claims Loaded: {len(df_claims)}")
    print(f" Meds Loaded: {len(df_meds)}")
    print(f" Master Lists Loaded")
else:
    print(" Error: Step 2 file not found.")

--- STEP 3: FEATURE ENGINEERING ---
Loading datasets...


 Claims Loaded: 314189
 Meds Loaded: 955045
 Master Lists Loaded


### Step 2: Process Medications → `MED_*` binary flags
Joins every medication row to its drug classification via the Medication Master, then pivots on `CLAIM_CODE` to produce one binary flag per drug class. The `MED_` prefix is part of the project's naming contract — downstream code discovers these columns by prefix match.

GLP-1 variants (e.g. injectable) are collapsed into a single `MED_GLP_1_RECEPTOR_AGONISTS` column so the aggregation rule at the patient level stays clean.

In [2]:
print(" Processing Medications...")

# 1. Identify Drug Code Column
med_col = "DRUG_CODE" if "DRUG_CODE" in df_meds.columns else "ITEM_CODE"
df_meds[med_col] = df_meds[med_col].astype(str).str.strip()

# 2. Merge with Master
df_med_class = pd.merge(
    df_meds, 
    df_med_master[["diabetes_drug_code", "drug_classification"]], 
    left_on=med_col, 
    right_on="diabetes_drug_code", 
    how="left"
).dropna(subset=["drug_classification"])

# 3. Pivot to Flags
df_med_class["Present"] = 1
df_med_flags = df_med_class.pivot_table(
    index="CLAIM_CODE", 
    columns="drug_classification", 
    values="Present", 
    aggfunc="max", 
    fill_value=0
)

# 4. Rename Columns
new_cols = []
for c in df_med_flags.columns:
    clean_name = str(c).upper().replace(' ', '_').replace('(', '').replace(')', '').replace('-', '_')
    new_cols.append(f"MED_{clean_name}")
df_med_flags.columns = new_cols

# 5. Deduplicate Columns (Transpose -> Group -> Transpose)
df_med_flags = df_med_flags.T.groupby(level=0).max().T

# Combine GLP-1 Columns
glp_cols = [c for c in df_med_flags.columns if "GLP_1" in c]

if len(glp_cols) > 0:
    print(f"   -> Found GLP-1 columns to merge: {glp_cols}")
    # Create ONE combined column (Max value: if either is 1, result is 1)
    df_med_flags["MED_GLP_1_RECEPTOR_AGONISTS"] = df_med_flags[glp_cols].max(axis=1)
    
    # Identify columns to drop (The "Injectable" one, or any duplicates)
    # We keep only the clean "MED_GLP_1_RECEPTOR_AGONISTS"
    cols_to_drop = [c for c in glp_cols if c != "MED_GLP_1_RECEPTOR_AGONISTS"]
    
    if cols_to_drop:
        df_med_flags.drop(columns=cols_to_drop, inplace=True)
        print(f"   -> Merged and dropped: {cols_to_drop}")
# ----------------------------------------

# Reset index so CLAIM_CODE becomes a column again
df_med_flags = df_med_flags.reset_index()

print(f" Created Medication Flags for {len(df_med_flags)} visits.")
display(df_med_flags.head())

 Processing Medications...


   -> Found GLP-1 columns to merge: ['MED_GLP_1_RECEPTOR_AGONISTS', 'MED_GLP_1_RECEPTOR_AGONISTS_INJECTABLE']
   -> Merged and dropped: ['MED_GLP_1_RECEPTOR_AGONISTS_INJECTABLE']
 Created Medication Flags for 218325 visits.


,CLAIM_CODE,MED_ALPHA_GLUCOSIDE_INHIBITORS,MED_BIGUANIDE,MED_COMBINATION_DRUG,MED_DPP_4_INHIBITORS,MED_GLP_1_RECEPTOR_AGONISTS,MED_INSULIN_THERAPY,MED_MEGLITINIDES,MED_SGLT_2_INHIBITORS,MED_SULPHONYLUREAS,MED_THIAZOLIDINEDIONES
0,2023010711105362,0,0,1,0,0,0,0,0,0,0
1,2023010714491198,0,0,1,0,0,0,0,1,0,0
2,2023010910095952,0,0,1,0,0,1,0,1,1,0
3,2023010910353798,0,0,1,0,0,0,0,0,0,0
4,2023010910401759,0,0,1,0,1,0,0,0,1,0


### Step 3: Process Complications → `COMP_*` binary flags
Melts every `DIAG_*` column into a long format, joins each code to its systemic complication category (retinopathy / nephropathy / neuropathy / macrovascular / diabetic foot / other specified), and pivots back to wide with one `COMP_*` flag per category at the claim level. The custom `if/elif` rename block normalises messy master-list category names into the five canonical `COMP_*` column names used by the ML layer.

In [3]:
print(" Processing Complications...")

# 1. Identify Diagnosis Columns
diag_cols = [c for c in df_claims.columns if c.startswith("DIAG_")]
print(f"   -> Scanning columns: {diag_cols}")

# 2. Melt to Long Format
df_long = df_claims.melt(
    id_vars="CLAIM_CODE", 
    value_vars=diag_cols, 
    value_name="DIAG_CODE"
).dropna()

# Data Cleaning
# Clean Claims Codes
df_long["DIAG_CODE"] = df_long["DIAG_CODE"].astype(str).str.strip().str.upper()

# Clean Master List Codes & Categories
df_comp_master["diabetes_diagnosis_code"] = df_comp_master["diabetes_diagnosis_code"].astype(str).str.strip().str.upper()
df_comp_master["systemic"] = df_comp_master["systemic"].astype(str).str.strip().str.upper()
# -----------------------

# 3. Merge with Master List
df_comp_class = pd.merge(
    df_long, 
    df_comp_master[["diabetes_diagnosis_code", "systemic"]], 
    left_on="DIAG_CODE", 
    right_on="diabetes_diagnosis_code", 
    how="left"
).dropna(subset=["systemic"])

# 4. Pivot to Flags
df_comp_class["Present"] = 1
df_comp_flags = df_comp_class.pivot_table(
    index="CLAIM_CODE", 
    columns="systemic", 
    values="Present", 
    aggfunc="max", 
    fill_value=0
)

# 5. RENAME COLUMNS (Custom Mapping based on your Screenshot)
new_cols = []
for c in df_comp_flags.columns:
    # 1. Basic Clean: Uppercase, replace spaces/slashes/dashes with underscore
    raw_name = str(c).upper().strip()
    
    # 2. Specific Mappings based on your file
    if "RETINOPATHY" in raw_name:
        clean_name = "RETINOPATHY"
    elif "NEPHROPATHY" in raw_name:
        clean_name = "NEPHROPATHY"
    elif "NEUROPATHY" in raw_name:
        clean_name = "NEUROPATHY"
    elif "CIRCULATORY" in raw_name or "MACROVASCULAR" in raw_name:
        clean_name = "MACROVASCULAR"
    elif "FOOT" in raw_name or "LIMB" in raw_name:
        clean_name = "DIABETIC_FOOT"
    elif "OTHER" in raw_name:
        clean_name = "OTHER_SPECIFIED"
    else:
        # Fallback for anything else: Remove weird characters
        clean_name = raw_name.replace('(', '').replace(')', '').replace('/', '_').replace(' ', '_')

    new_cols.append(f"COMP_{clean_name}")

df_comp_flags.columns = new_cols

# 6. Deduplicate Columns (Transpose -> Group -> Transpose)
# This ensures that if "(Eye) Retinopathy" and "Retinopathy" both exist, they merge.
df_comp_flags = df_comp_flags.T.groupby(level=0).max().T

# Reset index
df_comp_flags = df_comp_flags.reset_index()

print(f" Created Complication Flags for {len(df_comp_flags)} visits.")
print(f"   -> Columns: {df_comp_flags.columns.tolist()}")

 Processing Complications...
   -> Scanning columns: ['DIAG_1', 'DIAG_2', 'DIAG_3', 'DIAG_4']


 Created Complication Flags for 313868 visits.
   -> Columns: ['CLAIM_CODE', 'COMP_DIABETIC_FOOT', 'COMP_MACROVASCULAR', 'COMP_NEPHROPATHY', 'COMP_NEUROPATHY', 'COMP_OTHER_SPECIFIED', 'COMP_RETINOPATHY']


### Step 4: Merge everything and save
Left-joins the `MED_*` and `COMP_*` flag frames back onto the base wide claims (on `CLAIM_CODE`), fills missing flags with 0, and writes `step3_claims_meds_and_comps.xlsx` — the per-visit feature matrix that Step 4 (patient aggregation) will squash to one row per patient.

In [4]:
print(" Merging Everything...")

# 1. Start with the Base Claims (Visits)
df_final = df_claims.copy()

# 2. Join Medications (Left Join = Keep all visits)
df_final = pd.merge(df_final, df_med_flags, on="CLAIM_CODE", how="left")

# 3. Join Complications
df_final = pd.merge(df_final, df_comp_flags, on="CLAIM_CODE", how="left")

# 4. Fill NaNs with 0 for the new flag columns
# (If no match found, it means they didn't have that drug/complication)
flag_cols = [c for c in df_final.columns if c.startswith("MED_") or c.startswith("COMP_")]
df_final[flag_cols] = df_final[flag_cols].fillna(0).astype(int)

# 5. Save
print(f"💾 Saving to: {output_file}")
df_final.to_excel(output_file, index=False)
print(" Done. Step 3 Complete.")

 Merging Everything...


💾 Saving to: C:\Users\thira\Documents\GitHub\project-d\dataPreprocessing\source code\..\csv\step3_claims_meds_and_comps.xlsx


 Done. Step 3 Complete.
